### Notebook startup check

A small execution check to confirm that the notebook kernel is running. It does not affect the dataset or model.

In [2]:
print("Hello")

Hello


# CrowdSense AI: Crowd-Count Prediction

This notebook cleans a synthetic crowd dataset, builds features for predicting `Crowd_Count`, compares several regression models, and saves the selected XGBoost pipeline for reuse. Run the cells from top to bottom because later sections use the cleaned data, engineered features, and fitted preprocessor created earlier.

## 1. Import dependencies

Load data-handling, preprocessing, model-training, evaluation, and model-export libraries.

In [39]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score,mean_squared_error


## 2. Load and inspect the dataset

Read the master CSV, export its first five records as a small shareable subset, and inspect the schema and dimensions. This establishes the available features and confirms that the file was read correctly.

In [4]:
df=pd.read_csv("CrowdSense_AI_Master_Synthetic_Dataset.csv")

In [5]:
subset=df.head()

In [6]:
subset.to_csv("subsetCrowd.csv",index=False)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Record_ID                  1500 non-null   int64  
 1   Date                       1500 non-null   str    
 2   Time                       1500 non-null   str    
 3   City                       1500 non-null   str    
 4   Place                      1500 non-null   str    
 5   Region                     1500 non-null   str    
 6   Latitude                   1500 non-null   float64
 7   Longitude                  1500 non-null   float64
 8   Crowd_Count                1500 non-null   int64  
 9   Venue_Capacity             1500 non-null   int64  
 10  Venue_Area_km2             1500 non-null   float64
 11  Crowd_Density_per_km2      1500 non-null   float64
 12  Capacity_Utilization_pct   1500 non-null   float64
 13  Weather                    1500 non-null   str    
 14  Tem

In [8]:
df.shape

(1500, 32)

## 3. Check basic data quality

Count fully duplicated rows, remove repeated `Record_ID` values so each observation is unique, and review missing values before cleaning and modeling.

In [10]:
duplicates_count = df.duplicated().sum()

In [11]:
print(duplicates_count)

0


In [12]:
df = df.drop_duplicates(subset=['Record_ID'])

In [13]:
print(df.isnull().sum())

Record_ID                    0
Date                         0
Time                         0
City                         0
Place                        0
Region                       0
Latitude                     0
Longitude                    0
Crowd_Count                  0
Venue_Capacity               0
Venue_Area_km2               0
Crowd_Density_per_km2        0
Capacity_Utilization_pct     0
Weather                      0
Temperature_C                0
Humidity_pct                 0
Rainfall_mm                  0
Wind_Speed_kmh               0
Day_of_Week                  0
Holiday                      0
Event                        0
Event_Type                   0
Week_of_Year                 0
Special_Features             0
Transportation_Type          0
Peak_Hour                    0
Historical_Average_Crowd     0
Historical_Peak_Crowd        0
Historical_Incident_Count    0
Previous_Overcrowding        0
Risk_Score                   0
Risk_Level                   0
dtype: i

Standardize Text & Categorical Consistency
In real-world data, spelling variations or inconsistent casing (e.g., 'Clear', 'clear ', 'CLEAR') cause the model to treat them as distinct categories.

In [14]:
text_cols = ['City', 'Place', 'Region', 'Weather', 'Day_of_Week', 'Event', 'Event_Type', 'Special_Features', 'Transportation_Type']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

Capacity & Crowd Bounds:Crowd_Count must be $\ge 0$.Venue_Capacity and Venue_Area_km2 must be $> 0$.

In [15]:
df = df[(df['Crowd_Count'] >= 0) & (df['Venue_Capacity'] > 0) & (df['Venue_Area_km2'] > 0)]

Weather Physical Ranges:Humidity_pct must be within $[0, 100]$.Rainfall_mm must be $\ge 0$.Wind_Speed_kmh must be $\ge 0$.

In [16]:
df = df[(df['Humidity_pct'].between(0, 100)) & (df['Rainfall_mm'] >= 0) & (df['Wind_Speed_kmh'] >= 0)]

Geographical Coordinate Bounds: Validate latitude and longitude values against valid global ranges: latitude $[-90, 90]$ and longitude $[-180, 180].

In [17]:
df = df[df['Latitude'].between(-90, 90) & df['Longitude'].between(-180, 180)]

Validate Date into a standard datetime format (YYYY-MM-DD)

In [18]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

Validate Time string format (ensuring HH:MM format without extra characters):

In [19]:
df['Time'] = df['Time'].astype(str).str.strip()

## 4. Engineer calendar and time features

Convert the validated date and time fields into model-friendly features. `Month` and `Day` capture calendar effects; `Decimal_Hour` represents time numerically; and `hour_sin`/`hour_cos` encode the 24-hour cycle without making midnight and 23:00 appear far apart.

In [20]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

In [21]:
time_parts = df['Time'].astype(str).str.split(':', expand=True).astype(int)
df['Decimal_Hour'] = time_parts[0] + (time_parts[1] / 60.0)

In [22]:
df['hour_sin'] = np.sin(2 * np.pi * df['Decimal_Hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['Decimal_Hour'] / 24.0)

## 5. Define features and prevent target leakage

Set `Crowd_Count` as the prediction target. Remove identifiers, raw date/time fields, regional metadata, and fields calculated from the crowd count or risk outcome; retaining those derived fields would leak the answer into training and inflate evaluation results.

In [23]:
leakage_and_metadata = [
    'Record_ID', 'Date', 'Time', 'Decimal_Hour', 'Region',
    'Crowd_Count',                # Target variable
    'Crowd_Density_per_km2',      # Derived from Crowd_Count
    'Capacity_Utilization_pct',   # Derived from Crowd_Count
    'Risk_Score',                 # Derived target
    'Risk_Level'                  # Derived target
]

In [24]:
X = df.drop(columns=[col for col in leakage_and_metadata if col in df.columns])
y = df['Crowd_Count']

In [25]:
categorical_features = [
    'City', 'Place', 'Weather', 'Day_of_Week',
    'Event', 'Event_Type', 'Special_Features', 'Transportation_Type'
]

categorical_features = [col for col in categorical_features if col in X.columns]

In [26]:
numerical_features = [col for col in X.columns if col not in categorical_features]

## 6. Preprocess numeric and categorical features

Scale numeric inputs and one-hot encode categorical inputs. `handle_unknown='ignore'` lets the pipeline safely score a future record containing a category that was not present during training.

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

## 7. Create training and test sets

Reserve 20% of the data for an unseen test set. The fixed `random_state=42` makes the split reproducible; preprocessing is fitted only on training data and then applied to test data to avoid information leakage.

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [31]:
print("Preprocessing complete!")
print(f"X_train transformed shape: {X_train_processed.shape}")
print(f"X_test transformed shape: {X_test_processed.shape}")

Preprocessing complete!
X_train transformed shape: (1200, 103)
X_test transformed shape: (300, 103)


## 8. Train and evaluate the Random Forest baseline

Train a 100-tree Random Forest regressor on the processed training features, predict crowd counts for the holdout data, and report MAE, RMSE, and R². Lower MAE/RMSE indicate smaller prediction errors; higher R² indicates more variation explained.

In [34]:
model = RandomForestRegressor(
    n_estimators=100,      # Number of decision trees
    random_state=42,       # Ensures reproducible results
    n_jobs=-1              # Uses all CPU cores for faster training
)

In [35]:
model.fit(X_train_processed, y_train)

print("Model training complete!")

Model training complete!


In [37]:
y_pred = model.predict(X_test_processed)

In [40]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [41]:
print(f"Mean Absolute Error (MAE): {mae:.2f} people")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} people")

Mean Absolute Error (MAE): 1379.47 people
Root Mean Squared Error (RMSE): 1880.06 people


In [42]:
if len(y_test) > 1:
    r2 = r2_score(y_test, y_pred)
    print(f"R-squared (R2 Score): {r2:.4f}")

R-squared (R2 Score): 0.9873


## 9. Convert a prediction into an operational risk assessment

Translate predicted crowd count into capacity utilization and a Low, Moderate, High, or Critical risk level. The function also returns an alert color and a recommended operational response for a sample test venue.

In [43]:
def assign_risk_level(predicted_crowd: float, venue_capacity: int) -> dict:
    utilization_pct = (predicted_crowd / venue_capacity) * 100

    if utilization_pct < 60.0:
        risk_level = "Low"
        alert_color = "Green"
        action = "Normal monitoring. Standard entry/exit flow."
    elif utilization_pct < 80.0:
        risk_level = "Moderate"
        alert_color = "Yellow"
        action = "Deploy traffic personnel; monitor queue buildup."
    elif utilization_pct <= 100.0:
        risk_level = "High"
        alert_color = "Orange"
        action = "Enforce crowd diversion; restrict incoming entry gates."
    else:
        risk_level = "Critical"
        alert_color = "Red"
        action = "Immediate action: open emergency exits, issue diversion alerts."

    return {
        "predicted_crowd": round(predicted_crowd),
        "venue_capacity": venue_capacity,
        "capacity_utilization_pct": round(utilization_pct, 2),
        "risk_level": risk_level,
        "alert_color": alert_color,
        "recommended_action": action
    }

# Test sample calculation using the first test item
sample_venue_capacity = int(X_test['Venue_Capacity'].iloc[0])
sample_prediction = float(y_pred[0])
risk_result = assign_risk_level(sample_prediction, sample_venue_capacity)

print("Sample Risk Assessment Output:")
for key, value in risk_result.items():
    print(f"  {key}: {value}")

Sample Risk Assessment Output:
  predicted_crowd: 31120
  venue_capacity: 50360
  capacity_utilization_pct: 61.8
  risk_level: Moderate
  alert_color: Yellow
  recommended_action: Deploy traffic personnel; monitor queue buildup.


## 10. Compare candidate models

Train Random Forest, XGBoost, and LightGBM using the same processed train/test data, then compare their MAE, RMSE, and R² scores. This provides an evidence-based basis for choosing the final model. XGBoost and LightGBM must be installed in the active notebook kernel.

In [47]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Define candidate models
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42, 
        n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=100, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42, 
        n_jobs=-1,
        verbose=-1
    )
}

# Train and evaluate each model
results = []

for name, model in models.items():
    # 1. Fit on training data
    model.fit(X_train_processed, y_train)
    
    # 2. Predict on unseen test data
    y_pred = model.predict(X_test_processed)
    
    # 3. Compute metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred) if len(y_test) > 1 else np.nan
    
    results.append({
        "Model": name,
        "MAE (Lower is Better)": round(mae, 2),
        "RMSE (Lower is Better)": round(rmse, 2),
        "R2 Score (Closer to 1 is Better)": round(r2, 4) if not np.isnan(r2) else "N/A (single test sample)"
    })

# Format comparison as a clean DataFrame
comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

        Model  MAE (Lower is Better)  RMSE (Lower is Better)  R2 Score (Closer to 1 is Better)
Random Forest                1379.47                 1880.06                            0.9873
      XGBoost                1365.68                 1862.54                            0.9876
     LightGBM                1414.57                 1988.87                            0.9858


## 11. Build the reusable production pipeline

Configure XGBoost as the chosen model and combine it with the preprocessor in one scikit-learn `Pipeline`. Bundling them ensures new raw input receives exactly the same transformations used during training.

In [48]:
best_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

In [49]:
crowdsense_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_model)
])

In [50]:
crowdsense_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 12. Export the trained pipeline

Save the fitted preprocessing-and-XGBoost pipeline as `crowdsense_xgboost_pipeline.pkl`. The exported file can be loaded by the application to make predictions from raw feature values without retraining.

In [51]:
joblib.dump(crowdsense_pipeline, 'crowdsense_xgboost_pipeline.pkl')
print("Complete pipeline exported successfully to 'crowdsense_xgboost_pipeline.pkl'!")

Complete pipeline exported successfully to 'crowdsense_xgboost_pipeline.pkl'!


Verification of the Model


In [53]:

# Load the saved pipeline
loaded_pipeline = joblib.load('crowdsense_xgboost_pipeline.pkl')

# Take a single raw row from X_test to simulate an API request
sample_input = X_test.iloc[[1]]
predicted_crowd = loaded_pipeline.predict(sample_input)[0]

# Venue capacity from the input
capacity = sample_input['Venue_Capacity'].values[0]
utilization = (predicted_crowd / capacity) * 100

# Compute risk tier
if utilization < 60:
    risk = "Low"
elif utilization < 80:
    risk = "Moderate"
elif utilization <= 100:
    risk = "High"
else:
    risk = "Critical"

print(f"Predicted Crowd: {round(predicted_crowd):,}")
print(f"Venue Capacity: {capacity:,}")
print(f"Capacity Utilization: {utilization:.2f}%")
print(f"Assigned Risk Level: {risk}")

Predicted Crowd: 69,250
Venue Capacity: 61,453
Capacity Utilization: 112.69%
Assigned Risk Level: Critical
